# Chapter 7 &mdash; The Formal NFA: $\delta$ Returns a Set

**Concept 4 of the Chapter 7 decomposition:** *The Formal NFA: $(Q,\Sigma,\delta,Q_0,F)$ with $\delta: Q\times\Sigma_\varepsilon\to{\cal P}(Q)$*

Three changes from the DFA tuple: $\delta$ returns a <i>set</i>, accepts $\varepsilon$, and the start is a set $Q_0$.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Formal-NFA-Tuple/Concept-Formal-NFA-Tuple.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


An NFA is $(Q,\Sigma,\delta,Q_0,F)$ with exactly three differences from a DFA:

* $\delta: Q\times\Sigma_\varepsilon \to \mathcal{P}(Q)$ &mdash; the result is a **set of
  states**, possibly empty;
* the input alphabet is $\Sigma_\varepsilon = \Sigma\cup\{\varepsilon\}$;
* the start is a **set** $Q_0 \subseteq Q$, not a single state.

Returning $\emptyset$ is how a token **dies**, so an NFA's $\delta$ is **total** in a
trivial sense &mdash; there is no need to totalize, and no black hole.

## 2. Definitions

### The five keys, NFA flavour

In [ ]:
N = md2mc('''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> F
''')
for k in ['Q', 'Sigma', 'Delta', 'Q0', 'F']:
    v = N[k]
    print("%-6s : %s" % (k, sorted(v) if isinstance(v, set) else v))

### $\delta$ as a table of **sets**

In [ ]:
def delta_table(N):
    for q in sorted(N["Q"]):
        for a in sorted(N["Sigma"]) + ['']:
            r = step_nfa(N, q, a)
            print("   delta(%-3s, %-3s) = %s" % (q, repr(a), sorted(r) if r else "{}  <- token dies"))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch7&nbsp;3.&nbsp;$\varepsilon$-Transitions, and Whether They Are Essential](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Epsilon-Transitions/Concept-Epsilon-Transitions.ipynb) &nbsp;&middot;&nbsp; [**Chapter 7** index](https://github.com/ganeshutah/Jove/blob/master/Chapter7/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;5.&nbsp;Simulating an NFA Without $\varepsilon$: Tracking the Set of Token Positions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Simulating-Without-Epsilon/Concept-Simulating-Without-Epsilon.ipynb)&nbsp;&rarr;

---

## 3. Tests

Every result is a **set** &mdash; sometimes empty, sometimes several states.

In [ ]:
delta_table(N)
assert all(isinstance(step_nfa(N, q, a), set)
           for q in N["Q"] for a in list(N["Sigma"]) + [''])

`Q0` is a set, and can hold more than one state.

In [ ]:
Multi = md2mc('''NFA
I1 : 0 -> F
I2 : 1 -> F
''')
print("Q0 =", sorted(Multi["Q0"]), " <- two initial states, legal for an NFA")
assert len(Multi["Q0"]) == 2
print("accepts '0'?", accepts_nfa(Multi, '0'), "  accepts '1'?", accepts_nfa(Multi, '1'))
assert accepts_nfa(Multi, '0') and accepts_nfa(Multi, '1')
print("\nThe same markdown as a DFA would be rejected: 'DFA with 2 starting states'.")

No totalization is needed: $\emptyset$ already means 'reject along this path'.

In [ ]:
print("delta(A, '0') from the first machine :", sorted(step_nfa(N, 'A', '0')))
print("delta(F, '0')                        :", step_nfa(N, 'F', '0'), " <- empty, no black hole needed")
assert step_nfa(N, 'F', '0') == set()

Building one with `mk_nfa` gives the same machine.

In [ ]:
N2 = mk_nfa(N["Q"], N["Sigma"], N["Delta"], N["Q0"], N["F"])
from itertools import product
assert all(accepts_nfa(N, ''.join(p)) == accepts_nfa(N2, ''.join(p))
           for k in range(8) for p in product('01', repeat=k))
print("mk_nfa reconstruction agrees on all strings up to length 7")

## 4. Exercises


1. Which of the three differences is the one that really creates nondeterminism?
2. Why does an NFA need no black-hole state?
3. Write a DFA's five-tuple as an NFA's five-tuple. What changes?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter7/Concept-Formal-NFA-Tuple')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')